# 🪞 MiraComfy

**Run [ComfyUI](https://github.com/Comfy-Org/ComfyUI) on Google Colab — modernized for the free tier (Tesla T4 · 15 GB VRAM).**

### 🚀 Quick start
1. **Runtime ▸ Change runtime type ▸ Hardware accelerator: T4 GPU** (free tier).
2. Run the cells top-to-bottom: **Runtime check → (optional) Mount Drive → Setup → Custom nodes → Models → Start**.
3. Open the `https://…trycloudflare.com` URL printed by the Start cell. Done.

### ✨ What's new in this refresh
- **No CUDA wheel roulette** — reuses Colab's preinstalled, CUDA-enabled PyTorch (Python 3.12 / torch 2.8+) instead of pulling 2023-era `cu117/cu118/cu121` indexes that used to downgrade or break torch.
- **Upstream URLs updated** — ComfyUI now lives at `Comfy-Org/ComfyUI`, ComfyUI-Manager at `Comfy-Org/ComfyUI-Manager`.
- **Fixed dead model links** — `runwayml/stable-diffusion-v1-5` was taken down in 2024; defaults now use verified, anonymously-downloadable files (official `Comfy-Org` SD 1.5 archive, SDXL base, VAEs).
- **Hybrid persistence** — the app installs to fast local disk while `models/` and `output/` are symlinked to Google Drive, so multi-GB downloads and generated images survive runtime recycles.
- **One smart launcher** — Cloudflare tunnel (recommended) or localtunnel fallback, with double-start protection and a stop/restart cell.

# 🚦 0 · Runtime Check

Run this first. On the free tier you want a **T4 GPU** runtime. If no GPU is found, use
**Runtime ▸ Change runtime type ▸ Hardware accelerator: T4 GPU**, then run this cell again.

In [ ]:
import subprocess, sys

def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()

print("MiraComfy environment check")
print("=" * 50)

gpu = sh("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader")
if gpu:
    fields = [x.strip() for x in gpu.split(",")]
    print(f"🖥️  GPU       : {fields[0]} ({fields[1]}), driver {fields[2]}")
else:
    print("🖥️  GPU       : ❌ NOT FOUND")

try:
    import torch
    print(f"🔥 PyTorch   : {torch.__version__} | CUDA available: {'✅' if torch.cuda.is_available() else '❌'}")
except ImportError:
    print("🔥 PyTorch   : will be verified after the Setup cell")

disk = sh("df -h /content | awk 'NR==2 {print $4}'")
print(f"💾 Free disk : {disk} on /content")
print(f"🐍 Python    : {sys.version.split()[0]}")
print("=" * 50)

if gpu:
    print("✅ Ready — continue with the next cell.")
else:
    print("⚠️  No GPU detected — image generation will be unusably slow on CPU.")
    print("    Fix: Runtime ▸ Change runtime type ▸ Hardware accelerator: T4 GPU")

# 🔗 1 · Mount Google Drive *(optional, recommended)*

With Drive mounted, your **models and generated images persist** across Colab runtime recycles
(the free tier wipes the VM every few hours). Skip this cell if you don't need persistence.

In [ ]:
#@markdown <br><center><img src='https://upload.wikimedia.org/wikipedia/commons/thumb/d/da/Google_Drive_logo.png/600px-Google_Drive_logo.png' height="50" alt="Gdrive-logo"/></center>
#@markdown <center><h3>Mount / unmount Google Drive</h3></center><br>
MODE = "MOUNT" #@param ["MOUNT", "UNMOUNT"]

from pathlib import Path

if MODE == "MOUNT":
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=True)
        DRIVE_ROOT = "/content/drive/MyDrive"
        assert Path(DRIVE_ROOT).exists(), "Drive mounted but MyDrive was not found"
        DRIVE_MOUNTED = True
        print(f"\n✅ Drive mounted → {DRIVE_ROOT}")
    except Exception as e:
        DRIVE_MOUNTED = False
        print(f"\n⚠️  Drive not mounted ({e}). Persistence will be disabled — you can still continue.")
elif MODE == "UNMOUNT":
    from google.colab import drive
    try:
        drive.flush_and_unmount()
    except ValueError:
        pass
    get_ipython().system("rm -rf /root/.config/Google/DriveFS")
    DRIVE_MOUNTED = False
    print("✅ Drive unmounted.")

# ⚙️ 2 · Setup & Update ComfyUI

| Option | Meaning |
|---|---|
| **Local disk** *(default)* | Fast installs & model loads. The app itself is wiped when Colab recycles the VM — Setup just re-clones (~1 min). |
| **Google Drive** | The whole install persists on Drive (slower I/O, and `git` can occasionally leave lock files — see troubleshooting at the bottom). |

When Drive is mounted and the install is local, `models/` and `output/` are **transparently symlinked to Drive**,
so checkpoints and images survive recycles while the app stays fast.

ℹ️ Colab already ships a CUDA-enabled PyTorch — this setup **reuses it**. No `xformers`/CUDA-index downgrades:
modern ComfyUI uses PyTorch SDPA attention by default, which performs well on the T4.

In [ ]:
INSTALL_LOCATION = "Local disk (fast, ephemeral)" #@param ["Local disk (fast, ephemeral)", "Google Drive (persistent, slower)"]
UPDATE_COMFY_UI = True #@param {type:"boolean"}
PERSIST_MODELS_TO_DRIVE = True #@param {type:"boolean"}
PERSIST_OUTPUT_TO_DRIVE = True #@param {type:"boolean"}

import os, shutil, subprocess, sys
from pathlib import Path

DRIVE_MOUNTED = globals().get("DRIVE_MOUNTED", False)
DRIVE_ROOT    = Path(globals().get("DRIVE_ROOT", "/content/drive/MyDrive"))
COMFY_REPO    = "https://github.com/Comfy-Org/ComfyUI.git"  # ComfyUI's new home (the old comfyanonymous URL redirects here)

if INSTALL_LOCATION.startswith("Local"):
    WORKSPACE = Path("/content/ComfyUI")
else:
    if not DRIVE_MOUNTED:
        raise RuntimeError("Google Drive is not mounted. Run the Mount cell above first, or pick 'Local disk'.")
    WORKSPACE = DRIVE_ROOT / "ComfyUI"
    PERSIST_MODELS_TO_DRIVE = PERSIST_OUTPUT_TO_DRIVE = False  # already persistent on Drive

os.makedirs(WORKSPACE, exist_ok=True)

# ── clone or update ─────────────────────────────────────────────
if not (WORKSPACE / "main.py").exists():
    print(f"-= Initial setup: cloning ComfyUI → {WORKSPACE} =-")
    r = subprocess.run(["git", "clone", "--depth", "1", COMFY_REPO, str(WORKSPACE)],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError("Clone failed:\n" + (r.stdout or "") + (r.stderr or ""))
elif UPDATE_COMFY_UI:
    print("-= Updating ComfyUI =-")
    r = subprocess.run(["git", "pull", "--ff-only"], cwd=WORKSPACE, capture_output=True, text=True)
    print((r.stdout or "").strip() or (r.stderr or "").strip() or "already up to date")
else:
    print("-= Reusing existing ComfyUI (update skipped) =-")

# ── dependencies: keep Colab's preinstalled PyTorch, no CUDA-index downgrades ──
print("-= Installing dependencies (reusing Colab's preinstalled PyTorch) =-")
os.chdir(WORKSPACE)
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
                   capture_output=True, text=True)
if r.returncode != 0:
    raise RuntimeError("pip install failed:\n" + (r.stdout or "") + (r.stderr or ""))

import torch
print(f"-= torch {torch.__version__} | CUDA available: {torch.cuda.is_available()} =-")
if not torch.cuda.is_available():
    print("⚠️  CUDA unavailable — did you pick a GPU runtime? (Runtime ▸ Change runtime type)")

# ── persist models/output to Drive via symlinks ─────────────────
MODEL_SUBDIRS = ["checkpoints", "vae", "loras", "unet", "diffusion_models", "clip",
                 "clip_vision", "text_encoders", "controlnet", "upscale_models",
                 "embeddings", "hypernetworks", "style_models"]

def _has_real_files(folder: Path) -> bool:
    # True if the folder contains anything heavier than repo placeholder files.
    if not folder.exists():
        return False
    return any(f.is_file() and f.stat().st_size > 4096 for f in folder.rglob("*"))

def persist_folder(name: str, move_existing: bool) -> None:
    src = WORKSPACE / name
    dst = DRIVE_ROOT / "ComfyUI" / name
    if src.is_symlink():
        print(f"   {name}/ → already persisted to Drive")
        return
    if _has_real_files(src) and not move_existing:
        print(f"   ⚠️  {name}/ already holds files locally — leaving it in place "
              "(move them to Drive manually if you want persistence)")
        return
    dst.mkdir(parents=True, exist_ok=True)
    if name == "models":
        for sub in MODEL_SUBDIRS:
            (dst / sub).mkdir(exist_ok=True)
    moved = 0
    if _has_real_files(src):  # e.g. old outputs → copy over before linking
        for f in [x for x in src.rglob("*") if x.is_file() and x.stat().st_size > 4096]:
            target = dst / f.relative_to(src)
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.move(str(f), str(target))
            moved += 1
    shutil.rmtree(src, ignore_errors=True)
    src.symlink_to(dst)
    extra = f" (moved {moved} existing items)" if moved else ""
    print(f"   {name}/ → persisted to {dst}{extra}")

if DRIVE_MOUNTED:
    print("-= Persisting to Google Drive =-")
    if PERSIST_MODELS_TO_DRIVE:
        persist_folder("models", move_existing=False)
    if PERSIST_OUTPUT_TO_DRIVE:
        persist_folder("output", move_existing=True)

print(f"\n✅ ComfyUI ready at {WORKSPACE}")

# 🧩 3 · Custom Nodes

**[ComfyUI-Manager](https://github.com/Comfy-Org/ComfyUI-Manager)** (now maintained by Comfy-Org —
the old `ltdrdata/…` URL still redirects) adds a node manager to the ComfyUI UI, so you can
install anything else with a click once the server is running.

Add more nodes as comma-separated **git URLs** in the form field below.

In [ ]:
INSTALL_COMFYUI_MANAGER = True #@param {type:"boolean"}
#@markdown Extra nodes as comma-separated git URLs (optional, may be empty):
EXTRA_CUSTOM_NODES = "" #@param {type:"string"}

import subprocess, sys
from pathlib import Path

if "WORKSPACE" not in globals():
    raise RuntimeError("Run the 'Setup & Update ComfyUI' cell first.")

def install_custom_node(url: str) -> None:
    name = url.rstrip("/").split("/")[-1].removesuffix(".git")
    node_dir = Path(WORKSPACE) / "custom_nodes" / name
    if node_dir.exists():
        print(f"   ↺ updating  {name}")
        subprocess.run(["git", "pull", "--ff-only"], cwd=node_dir, capture_output=True)
    else:
        print(f"   + cloning   {name}")
        r = subprocess.run(["git", "clone", "--depth", "1", url, str(node_dir)],
                           capture_output=True, text=True)
        if r.returncode != 0:
            print(f"   ❌ failed:  {name}\n{(r.stderr or '').strip()}")
            return
    reqs = node_dir / "requirements.txt"
    if reqs.exists():
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(reqs)],
                       capture_output=True)

if INSTALL_COMFYUI_MANAGER:
    print("-= ComfyUI-Manager =-")
    install_custom_node("https://github.com/Comfy-Org/ComfyUI-Manager.git")

for url in [u.strip() for u in EXTRA_CUSTOM_NODES.split(",") if u.strip()]:
    install_custom_node(url)

print("✅ Custom nodes done")

# 📦 4 · Models Download

Downloads use **aria2** (16 connections, resumable — safe to re-run).

**Secrets** *(sidebar 🔑 → “Secrets”, then enable notebook access)*:
- `CIVITAI_API_TOKEN` — most civitai.com downloads now require a token. Get one at
  civitai.com → account settings → API Keys.
- `HF_TOKEN` — only needed for *gated* Hugging Face models.

The default checkpoint below is a **verified, anonymously-downloadable** SD 1.5 build from the
official Comfy-Org archive — small (2.1 GB), fast on T4, works out of the box. Uncomment more as needed.

In [ ]:
import subprocess
from pathlib import Path
from google.colab import userdata

if "WORKSPACE" not in globals():
    raise RuntimeError("Run the 'Setup & Update ComfyUI' cell first.")

def _secret(name):
    try:
        value = userdata.get(name)
        return value if value else None
    except Exception:
        return None

CIVITAI_API_TOKEN = _secret("CIVITAI_API_TOKEN")
HF_TOKEN = _secret("HF_TOKEN")
print(f"Civitai token : {'✅ found' if CIVITAI_API_TOKEN else '— not set (only needed for civitai.com links)'}")
print(f"HF token      : {'✅ found' if HF_TOKEN else '— not set (only needed for gated HF models)'}")

print("-= Installing aria2 =-")
r = subprocess.run(["apt-get", "-y", "-qq", "install", "aria2"], capture_output=True)
if r.returncode != 0:  # fall back to refreshing the package index
    subprocess.run(["apt-get", "update", "-qq"], check=True, capture_output=True)
    subprocess.run(["apt-get", "-y", "-qq", "install", "aria2"], check=True, capture_output=True)

def download_model(url: str, filename: str = None, subfolder: str = "checkpoints") -> bool:
    # Resumable download into WORKSPACE/models/<subfolder>. Handles HF + Civitai.
    dest = Path(WORKSPACE) / "models" / subfolder
    dest.mkdir(parents=True, exist_ok=True)
    cmd = ["aria2c", "--console-log-level=error", "--summary-interval=0",
           "-c", "-x", "16", "-s", "16", "-k", "1M", "--auto-file-renaming=false",
           "-d", str(dest)]
    if "huggingface.co" in url:
        if filename is None:
            filename = url.split("/")[-1].split("?")[0]
        if HF_TOKEN:
            cmd += [f"--header=Authorization: Bearer {HF_TOKEN}"]
    elif "civitai.com" in url:
        if not CIVITAI_API_TOKEN:
            print("   ⚠️  no CIVITAI_API_TOKEN secret set — civitai may reject this download")
        else:
            url = url + ("&" if "?" in url else "?") + f"token={CIVITAI_API_TOKEN}"
    if filename:
        cmd += ["-o", filename]
    r = subprocess.run(cmd + [url])
    ok = r.returncode == 0
    print(f"   {'✅' if ok else '❌'} {filename or url}")
    return ok

print("\n-= Downloads =-")

# ── Default · Stable Diffusion 1.5 fp16 (official Comfy-Org archive · 2.1 GB) ──
download_model("https://huggingface.co/Comfy-Org/stable-diffusion-v1-5-archive/resolve/main/v1-5-pruned-emaonly-fp16.safetensors")

# ── Optional · uncomment what you need ──────────────────────────
# SD 1.5 full precision (4.3 GB)
# download_model("https://huggingface.co/Comfy-Org/stable-diffusion-v1-5-archive/resolve/main/v1-5-pruned-emaonly.safetensors")

# SDXL base 1.0 (6.9 GB — runs fine on T4, roughly 2× slower per image than SD1.5)
# download_model("https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0/resolve/main/sd_xl_base_1.0.safetensors")

# VAE for SD 1.x (fixes washed-out colors on some merges)
# download_model("https://huggingface.co/stabilityai/sd-vae-ft-mse-original/resolve/main/vae-ft-mse-840000-ema-pruned.safetensors", subfolder="vae")

# SDXL VAE (fp16-safe, no more black images)
# download_model("https://huggingface.co/madebyollin/sdxl-vae-fp16-fix/resolve/main/sdxl_vae.safetensors", subfolder="vae")

# ── Civitai examples (need CIVITAI_API_TOKEN; grab fresh links on civitai.com) ──
# download_model("https://civitai.com/api/download/models/2071650")   # CyberRealistic Pony
# download_model("https://civitai.com/api/download/models/288982")    # Juggernaut v8

print("\nDone. Use the next cell to list what you have.")

In [ ]:
# List everything in the models tree
from pathlib import Path

def human(n):
    for unit in ["B", "KB", "MB", "GB", "TB"]:
        if n < 1024:
            return f"{n:.2f} {unit}"
        n /= 1024
    return f"{n:.2f} PB"

models_dir = Path(WORKSPACE) / "models"
total, count = 0, 0
print(f"Models in {models_dir}\n" + "=" * 64)
for f in sorted(models_dir.rglob("*")):
    if f.is_file() and not f.name.startswith("put_") and f.stat().st_size > 4096:
        print(f"{human(f.stat().st_size):>12}  {f.relative_to(models_dir)}")
        total += f.stat().st_size
        count += 1
print("=" * 64)
print(f"{human(total):>12}  TOTAL ({count} files)" if count else "No models yet — run the download cell above.")

# ▶️ 5 · Start ComfyUI

- **Cloudflare** *(recommended)* — free quick tunnel, no account, prints a `…trycloudflare.com` URL.
- **Localtunnel** — fallback if Cloudflare misbehaves; its “tunnel password” is the IP printed above the URL.
- **None** — no tunnel; the server only listens inside the Colab VM (API use).

⚠️ The tunnel URL is **public** while the cell runs — anyone with the link can reach your ComfyUI.
💡 Handy `EXTRA_ARGS`: `--fast fp16_accumulation` (faster on T4), `--lowvram`, `--dont-print-server`.

In [ ]:
TUNNEL = "Cloudflare" #@param ["Cloudflare", "Localtunnel", "None (local only)"]
EXTRA_ARGS = "" #@param {type:"string"}
PORT = 8188 #@param {type:"integer"}

import os, shutil, socket, subprocess, threading, time, urllib.request

if "WORKSPACE" not in globals():
    raise RuntimeError("Run the 'Setup & Update ComfyUI' cell first.")

# refuse to double-start
probe = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
already_running = probe.connect_ex(("127.0.0.1", PORT)) == 0
probe.close()
if already_running:
    raise RuntimeError(f"Port {PORT} is busy — ComfyUI is probably already running. Use the Stop cell below first.")

if TUNNEL == "Cloudflare" and not shutil.which("cloudflared"):
    print("-= Installing cloudflared =-")
    subprocess.run("wget -q -O /tmp/cloudflared.deb https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb"
                   " && dpkg -i /tmp/cloudflared.deb", shell=True, check=True)
elif TUNNEL == "Localtunnel" and not shutil.which("lt"):
    print("-= Installing localtunnel =-")
    subprocess.run("npm install -g localtunnel", shell=True, check=True)

def _wait_and_tunnel(port):
    while True:
        s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        up = s.connect_ex(("127.0.0.1", port)) == 0
        s.close()
        if up:
            break
        time.sleep(0.5)
    print("\n✅ ComfyUI is up — starting tunnel (URL appears below)…")
    if TUNNEL == "Cloudflare":
        p = subprocess.Popen(["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{port}"],
                             stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        for raw in p.stderr:
            text = raw.decode()
            if "trycloudflare.com" in text:
                url = text[text.find("http"):].split()[0].strip(" |")
                print(f"\n🌍 ComfyUI URL: {url}\n")
    else:
        ip = urllib.request.urlopen("https://ipv4.icanhazip.com").read().decode().strip()
        print(f"🔒 localtunnel password (enter this when the page asks): {ip}")
        p = subprocess.Popen(["lt", "--port", str(port)], stdout=subprocess.PIPE)
        for raw in p.stdout:
            print(raw.decode(), end="")

if TUNNEL != "None (local only)":
    threading.Thread(target=_wait_and_tunnel, daemon=True, args=(PORT,)).start()
else:
    print("No tunnel started — ComfyUI only listens inside the Colab VM.")

os.chdir(WORKSPACE)
get_ipython().system(f"python main.py --listen 127.0.0.1 --port {PORT} --disable-auto-launch {EXTRA_ARGS}".strip())

In [ ]:
# 🛑 Stop ComfyUI + tunnels (then re-run the Start cell to relaunch)
import subprocess

for pattern in ["main.py", "cloudflared tunnel", "lt --port"]:
    subprocess.run(["pkill", "-f", pattern], capture_output=True)
print("🛑 Stopped. Outputs live in WORKSPACE/output (→ Drive if persistence is on).")

# 💡 Free-tier notes & troubleshooting

**Free tier reality:** T4 · 15 GB VRAM · ~12 GB RAM · ~75 GB disk. SD 1.5 is quick; SDXL works well
(~1 min per 1024² image); larger architectures (Flux, video models) only make sense quantized (GGUF/FP8).

| Symptom | Fix |
|---|---|
| Runtime wiped everything except Drive | Expected — free VMs are recycled. That's why `models/` and `output/` symlink to Drive. |
| Idle disconnects | Keep the tab open/interact; Colab kills idle GPU sessions. |
| Dependency/torch errors after an update | `Runtime ▸ Disconnect and delete runtime`, then run all cells again on a clean VM. |
| A model link 404s | Grab a fresh link (civitai → *Download* → copy link) and add it to the Models cell. |
| Cloudflare tunnel hangs | Stop the cell, retry, or switch `TUNNEL = "Localtunnel"`. |
| Drive install: `git` complains about lock files | Delete `ComfyUI/.git/index.lock` on Drive and re-run Setup. |
| First image takes forever | Normal — the model loads from Drive/disk into VRAM on first use. |

Generated images → `WORKSPACE/output` (persisted to Drive when mounted). The workflow **Templates**
browser is built into the ComfyUI sidebar — no extra nodes needed for the default SD 1.5 workflow.